# Estimate a threshold curve with Stim + PyMatching

Companion notebook to **Lattice Atlas → The Surface Code Lab**.

The browser Lab uses an ideal-check, i.i.d. data-Pauli toy. Here you use **Stim** and **PyMatching** to estimate logical failure for one generated circuit-level depolarizing model. The resulting crossing belongs to this circuit, decoder, distances, sampling budget, and noise definition; it is not a universal surface-code constant.

Prerequisite topics: *surface code*, *syndrome extraction circuits*, *decoding/MWPM*, *fault tolerance & thresholds*.

In [ ]:
%pip install -q stim pymatching matplotlib numpy

## 1 · Build a noisy surface-code memory circuit

`stim.Circuit.generated` produces the standard rotated memory-Z experiment — the same circuit family as the Lab's **Download .stim** button, with circuit-level noise: every gate, reset, and measurement can fail.

In [ ]:
import stim
import pymatching
import numpy as np

circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    distance=3,
    rounds=3,
    after_clifford_depolarization=0.005,
    after_reset_flip_probability=0.005,
    before_measure_flip_probability=0.005,
    before_round_data_depolarization=0.005,
)
print(f"{circuit.num_qubits} qubits, {circuit.num_detectors} detectors, {circuit.num_observables} observable")

## 2 · Sample and decode

The pipeline every decoding paper uses:

1. The circuit's **detector error model** (DEM) lists every possible fault and which detectors it flips — this is the decoder's map of the world.
2. PyMatching builds its matching graph straight from the DEM.
3. Stim samples detection events; PyMatching predicts the logical observable; disagreements are **logical errors**.

In [ ]:
def logical_error_result(distance: int, p: float, shots: int):
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        distance=distance,
        rounds=distance,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p,
    )
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    sampler = circuit.compile_detector_sampler()
    detections, observables = sampler.sample(shots, separate_observables=True)
    predictions = matcher.decode_batch(detections)
    errors = int(np.sum(np.any(predictions != observables, axis=1)))
    return errors / shots, errors

quick_rate, quick_failures = logical_error_result(3, 0.005, 5000)
print(f"d=3, p=0.5% → {quick_rate:.5f} ({quick_failures}/5000 failures)")

## 3 · The sweep

Sweep several finite distances under the generated circuit-level noise model. Do not infer a threshold from one pair of curves or quote a crossing without its circuit, decoder, distances, and uncertainty.

(A few minutes at these shot counts; lower `SHOTS` for a quick look.)

In [ ]:
import matplotlib.pyplot as plt

PS = [0.002, 0.004, 0.006, 0.008, 0.010, 0.012, 0.015]
DISTANCES = [3, 5, 7]
SHOTS = 20_000

results = {d: [logical_error_result(d, p, SHOTS) for p in PS] for d in DISTANCES}

def wilson_interval(failures, shots, z=1.96):
    phat = failures / shots
    denom = 1 + z*z/shots
    center = (phat + z*z/(2*shots)) / denom
    half = z * np.sqrt(phat*(1-phat)/shots + z*z/(4*shots*shots)) / denom
    return max(0.0, center-half), min(1.0, center+half)

fig, ax = plt.subplots(figsize=(7, 5))
for d, color in zip(DISTANCES, ["#0891B2", "#8B5CF6", "#D97706"]):
    rates = [item[0] for item in results[d]]
    failures = [item[1] for item in results[d]]
    bounds = [wilson_interval(f, SHOTS) for f in failures]
    # On a log axis, show a zero-failure cell at its 95% upper bound with a limit arrow.
    plotted = [r if f else hi for r, f, (_, hi) in zip(rates, failures, bounds)]
    lower = [y-lo if f else y/2 for y, f, (lo, _) in zip(plotted, failures, bounds)]
    upper = [hi-y if f else 0 for y, f, (_, hi) in zip(plotted, failures, bounds)]
    ax.errorbar(PS, plotted, yerr=[lower, upper], uplims=[f == 0 for f in failures],
                marker="o", color=color, label=f"d={d}", capsize=3)
ax.set_yscale("log")
ax.set_xlabel("physical error rate p")
ax.set_ylabel("logical error rate")
ax.set_title("Rotated surface code memory, circuit-level noise (Stim + PyMatching)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 4 · Read your plot like the papers do

- **Suppression evidence**: compare distances with their Wilson intervals. A fitted crossing is a finite-size estimate for this generated model, not a proof of a hardware threshold.
- **Zero observed failures** means an upper bound, not zero logical error; the plot marks the 95% Wilson upper limit.
- If you compute Λ = ε(d)/ε(d+2), retain raw failures/shots and intervals. Google's Λ ≈ 2.14 used a different physical device, task, and decoder, so it is context rather than a like-for-like benchmark.

### Keep going
- Load your own Lab export: `stim.Circuit(open("surface_code_d5_p0.080.stim").read())` and decode it with the same pipeline.
- Swap in `"surface_code:rotated_memory_x"` — why are the curves (almost) the same?
- Try `rounds=1` vs `rounds=d` — watch measurement noise destroy the single-round code and rediscover *why* syndromes are repeated.
- For serious sweeps, use `sinter` (Stim's Monte-Carlo harness) — it parallelizes and handles statistics properly.
- Paste any circuit into **Crumble** (https://algassert.com/crumble) to step through it tick by tick.